# yt-clips — Colab Worker
1. Runtime → T4 GPU
2. Add secrets to Colab Secrets (left panel): `OPENROUTER_API_KEY`, `GROQ_API_KEY`
3. Run the cell below

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

def sh(cmd, timeout=120):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if r.returncode and r.stderr: print(r.stderr.strip()[-300:])
    return r

print("="*55)
print("  COLAB WORKER SETUP")
print("="*55)

# ── Mount Drive ──
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ── cd to yt-clips on Drive ──
DRIVE_DIR = None
for p in ["/content/drive/MyDrive/yt-clips", "/content/drive/My Drive/yt-clips"]:
    if Path(p).exists(): DRIVE_DIR = p; break
if not DRIVE_DIR:
    raise SystemExit("yt-clips folder not found on Drive!")
os.chdir(DRIVE_DIR)
sys.path.insert(0, DRIVE_DIR)
print(f"PWD: {DRIVE_DIR}")

# ── Load secrets ──
for line in open(".env") if Path(".env").exists() else []:
    line = line.strip()
    if line and "=" in line and not line.startswith("#"):
        k, v = line.split("=", 1); os.environ[k.strip()] = v.strip()
try:
    from google.colab import userdata
    for k in ["OPENROUTER_API_KEY", "GROQ_API_KEY", "GOOGLE_API_KEY"]:
        v = userdata.get(k)
        if v: os.environ[k] = v
except: pass

# ── Install system deps ──
print("Installing system deps...")
sh("apt-get update -qq && apt-get install -y -qq aria2 ffmpeg >/dev/null 2>&1")

# ── Install deno (JS runtime for yt-dlp) ──
print("Installing deno...")
sh("curl -fsSL https://deno.land/x/install/install.sh | sh 2>&1 | tail -3")
DENO_BIN = os.path.expanduser("~/.deno/bin")
os.environ["PATH"] = DENO_BIN + ":" + os.environ.get("PATH", "")
print(f"deno: {sh('deno --version 2>&1 | head -1').stdout.strip()}")

# ── Install torch ──
print("Installing torch+cu121...")
sh("pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121", 300)

# ── Install pipeline deps ──
print("Installing pipeline deps...")
sh("pip install -q yt-dlp faster-whisper youtube-transcript-api "
   "rich PyYAML opencv-python-headless numpy "
   "filterpy scipy openai python-dotenv Pillow requests "
   "ultralytics gfpgan basicsr realesrgan "
   "google-api-python-client google-auth-httplib2 google-auth-oauthlib")

# ── Verify ──
import automation; print(f"automation v{automation.VERSION}")
import torch
print(f"CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")

# ── Dirs ──
for d in ["input","temp","transcripts","highlights","shorts","logs","photos"]:
    Path(d).mkdir(exist_ok=True)

# ── Kill old watchers + free port ──
sh("pkill -9 -f 'watcher.py' 2>/dev/null; sleep 1")
sh("fuser -k 5000/tcp 2>/dev/null; sleep 1")
sh("pkill -f serveo 2>/dev/null; pkill -f localhost.run 2>/dev/null")
time.sleep(2)

# ── Start watcher ──
watcher_path = str(Path(DRIVE_DIR) / "watcher.py")
sh(f"nohup {sys.executable} {watcher_path} > watcher.log 2>&1 &")
time.sleep(4)
ok = subprocess.run("curl -sf http://localhost:5000/health", shell=True, capture_output=True).returncode == 0
print(f"Watcher: {'OK' if ok else 'FAILED'}")
if not ok:
    log_content = open("watcher.log").read()[-500:] if Path("watcher.log").exists() else "no log"
    print(log_content)

# ── Start tunnel ──
def tunnel(cmd, wait=8):
    sh(cmd.format(L="tunnel.log"), 15); time.sleep(wait)
    if Path("tunnel.log").exists():
        for line in open("tunnel.log"):
            for w in line.split():
                w = w.strip().rstrip(",.;")
                if w.startswith("https://"):
                    Path("colab_url.txt").write_text(w); return w
    return None
url = tunnel("nohup ssh -o StrictHostKeyChecking=no -o ServerAliveInterval=30 -R 80:localhost:5000 serveo.net > {L} 2>&1 &")
if not url:
    print("serveo failed, trying localhost.run...")
    sh("pkill -f serveo 2>/dev/null; sleep 2")
    url = tunnel("nohup ssh -o StrictHostKeyChecking=no -o ServerAliveInterval=30 -R 80:localhost:5000 nokey@localhost.run > {L} 2>&1 &", 12)
print(f"Tunnel: {url or 'FAILED'}")

print("\n" + "="*55)
print("  WORKER ONLINE")
print(f"  Tunnel: {url}")
print("  Jobs are picked up from Drive (remote_job.json)")
print("  Or via tunnel (POST /job)")
print("="*55 + "\n")

# ── Dashboard loop ──
pos = 0
while True:
    time.sleep(15)
    log = Path("watcher.log")
    if log.exists():
        with open(log) as f:
            f.seek(pos)
            for l in f:
                if l.strip(): print(l.strip())
            pos = f.tell()
    mem = Path("/proc/meminfo")
    free = int(open(mem).read().split("MemFree:")[1].split()[0])/1e6 if mem.exists() else 0
    has_gpu = torch.cuda.is_available()
    gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.free","--format=csv,nounits,noheader"],
                        capture_output=True,text=True,timeout=5).stdout.strip() if has_gpu else "?"
    print(f"[RAM {free:.1f}GB | GPU {gpu[:30]} | Tunnel {'OK' if Path('colab_url.txt').exists() else '...'}]")
